# 📊 Auditoría del Mercado Laboral Tecnológico
### 🚀 *Pipeline Integral de Data Wrangling, Persistencia Relacional y Análisis Exploratorio SQL*

---

<div align="center">

![Python](https://img.shields.io/badge/Python-3.x-3776ab.svg?style=for-the-badge&logo=python&logoColor=white)
![Pandas](https://img.shields.io/badge/Pandas-Data%20Wrangling-150458.svg?style=for-the-badge&logo=pandas&logoColor=white)
![NumPy](https://img.shields.io/badge/NumPy-Vectorized-013243.svg?style=for-the-badge&logo=numpy&logoColor=white)
![SQLite](https://img.shields.io/badge/SQLite-Relational%20DB-003B57.svg?style=for-the-badge&logo=sqlite&logoColor=white)
![Status](https://img.shields.io/badge/Estado-Completado%20✅-2ea44f.svg?style=for-the-badge)

</div>

---

### 📌 Ficha Técnica del Proyecto

| Parámetro | Especificación Técnica | Detalle de Implementación |
| :--- | :--- | :--- |
| 📁 **Dataset Origen** | `DataAnalyst.csv` | Muestreo de ofertas laborales tech (Kaggle / Glassdoor) |
| 📐 **Volumen Inicial** | 2,253 registros × 16 columnas | Datos crudos con ruido textual y centinelas `-1` |
| 🛠️ **Stack Tecnológico** | Python 3 · Pandas · NumPy · SQLite3 | Enfoque end-to-end sin dependencias externas pesadas |
| 🎯 **Alcance Global** | Ingesta ➔ Limpieza Regex ➔ SQLite ➔ SQL EDA | Deuda técnica resuelta al 100% |

> 🎯 **Objetivo General:**  
> Diseñar y ejecutar un pipeline robusto, reproducible y documentado para la transformación de datos laborales no estructurados, culminando en un almacén relacional transaccional y la extracción de métricas de negocio mediante consultas SQL.

---

### 🗺️ Arquitectura del Pipeline

```text
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                                   PIPELINE WORKFLOW                                    │
└────────────────────────────────────────────────────────────────────────────────────────┘
  [ CSV Crudo ] ──► ( 1. Ingesta & Diagnóstico ) ──► Radiografía estructural y detección de fallas
                           │
                           ▼
                    ( 2. Data Wrangling )        ──► Regex, Type Casting, Imputación, Normalización
                           │
                           ▼
                    ( 3. Persistencia Relacional) ──► Túnel transaccional y volcado a SQLite (.db)
                           │
                           ▼
                    ( 4. SQL Analytics & EDA )   ──► KPIs de salarios, sectores y hubs geográficos
```


---
# 📗 FASE 1: Ingesta y Diagnóstico de Datos
> 🎯 **Objetivo:** Cargar el conjunto de datos en memoria (`DataFrame`) y aplicar una auditoría estructural sistemática para identificar ruido, tipos inconsistentes y nulos encubiertos antes de realizar cualquier transformación.

---

### 🔹 1.1 Importación del Ecosistema e Ingesta Cruda

Instanciamos los motores vectoriales **Pandas** (`pd`) y **NumPy** (`np`). Cargamos el archivo CSV crudo hacia una estructura tabular bidimensional en memoria volátil (`DataFrame`).

> 💡 **Detalle Técnico:** La ingesta directa en texto plano permite evaluar la inferencia automática de tipos que realiza Pandas de forma nativa.

In [ ]:
import pandas as pd
import numpy as np

# Ingesta: lectura del archivo de texto plano (CSV) hacia memoria
df = pd.read_csv('./data/DataAnalyst.csv')

### 🔹 1.2 Diagnóstico Estructural de Metadatos (`df.info()`)

Ejecutamos el método `.info()` para obtener la **radiografía técnica** del dataset:
* 📏 **Dimensionalidad:** Recuento total de filas (instancias) y columnas (variables).
* 🏷️ **Tipado primitivo:** Clasificación de cada columna (`int64`, `float64`, `object`).
* 🔍 **Valores No Nulos:** Densidad de completitud inmediata reportada por el motor.
* 💾 **Huella en Memoria:** Consumo de RAM en tiempo de ejecución.

> ⚠️ **Importancia Analítica:** Permite anticipar fallas de casting (ej. variables monetarias indexadas como texto plano) y detectar columnas redundantes antes de procesar.

In [ ]:
df.info()

#### 🔍 Hallazgos Críticos del Diagnóstico Inicial

| Anomalía Detectada | Evidencia Técnica | Severidad | Impacto en el Pipeline |
| :--- | :--- | :---: | :--- |
| **Índice Residual** | Columna `Unnamed: 0` | 🟡 Baja | Duplicidad de índice e incremento innecesario de memoria. |
| **Tipado Incorrecto** | `Salary Estimate` como `object` | 🔴 Alta | Imposibilita operaciones estadísticas y de agregación cuantitativa. |
| **Valores Centinela** | Presencia de `-1`, `"-1"`, `"Unknown"` | 🔴 Crítica | Falso positivo de completitud: el motor los computa como datos válidos. |

### 🔹 1.3 Saneamiento Estructural — Supresión de Columna Residual

Eliminamos la columna `Unnamed: 0` (artefacto serializado del proceso de web scraping) especificando el **eje horizontal de columnas** (`axis=1`).

```text
       ┌─────────── axis=0 (Filas / Operación Vertical)
       │
[ Matriz DataFrame ]
       │
       └─────────── axis=1 (Columnas / Operación Horizontal) ──► Drop 'Unnamed: 0'
```

In [ ]:
# Eliminación de la columna artefacto sobre el eje de columnas
df.drop('Unnamed: 0', axis=1, inplace=True)
df.head()

### 🔹 1.4 Aislamiento y Subsetting de Anomalías Visibles

Aplicamos **Subsetting Estructural** extrayendo una proyección matricial bidimensional (`df[['col1', 'col2', 'col3']]`) para inspeccionar cualitativamente los patrones de ruido en las columnas más vulnerables.

> 🔬 **Propósito:** Mapear la sintaxis de las cadenas de texto para diseñar los patrones de expresiones regulares requeridos.

In [ ]:
# Subsetting: aislamos las variables sospechosas
df[['Salary Estimate', 'Founded', 'Rating']].head()

#### 📋 Patrones de Ruido Identificados en `Salary Estimate`

```text
  Cadena Cruda:  "$37K-$66K (Glassdoor est.)"
                  ├── "$"                ──► Símbolo de divisa parásito
                  ├── "37" / "66"        ──► Límites cuantitativos (Min / Max)
                  ├── "K"                ──► Multiplicador de escala (×1,000)
                  ├── "-"                ──► Delimitador de rango
                  └── "(Glassdoor est.)" ──► Ruido contextual metadato
```

Adicionalmente, se identifica que `Company Name` concatena saltos de línea (`\n`) y el puntaje de calificación empresarial.

---

<div align="center">

| ✅ Hito Fase 1 | Diagnóstico y perfilado completado satisfactoriamente. El DataFrame está listo para Data Wrangling. |
| :--- | :--- |

</div>

---
# 📙 FASE 2: Limpieza y Normalización (Data Wrangling)
> 🎯 **Objetivo:** Ejecutar el saneamiento integral de datos — extracción vectorial con Regex, coerción y tipado numérico estricto, estandarización de centinelas a `NaN` y resolución metódica del déficit de información.

---

### 🔹 2.1 Extracción Vectorial de Límites Salariales mediante Regex

Implementamos un autómata de patrones con **Expresiones Regulares** aplicado vectorialmente sobre la serie textual mediante el accesor `.str.extract()`.

```text
Patrón de Captura:   \$(\d+)K-\$(\d+)K
                     │  │      │  │
                     │  │      │  └── (Grupo 2): Captura dígitos del salario Máximo
                     │  │      └───── Símbolo literal $ y delimitador de rango "-"
                     │  └──────────── (Grupo 1): Captura dígitos del salario Mínimo
                     └─────────────── Símbolo literal $ escapado
```

In [ ]:
# Extracción de los límites salariales mediante Regex
df[['Min_Salary', 'Max_Salary']] = df['Salary Estimate'].str.extract(r'\$(\d+)K-\$(\d+)K')

# Validación visual: comparamos la columna original con las nuevas
df[['Salary Estimate', 'Min_Salary', 'Max_Salary']].head()

### 🔹 2.2 Coerción de Tipos (Type Casting) y Reescalado Numérico

Mutamos las nuevas series desde `object` (cadena) hacia `float64` —tipo numérico compatible con `NaN`— y restituimos su magnitud multiplicando vectorialmente por **1,000** para reflejar el valor monetario real en dólares estadounidenses ($ USD).

In [ ]:
# Type Casting: str → float64, y escalado ×1000
df['Min_Salary'] = df['Min_Salary'].astype(float) * 1000
df['Max_Salary'] = df['Max_Salary'].astype(float) * 1000

# Verificación de tipos resultantes
df[['Min_Salary', 'Max_Salary']].dtypes

### 🔹 2.3 Ingeniería de Características (Feature Engineering)

Derivamos una variable agregada clave para la analítica de negocio: el **Salario Promedio** (`Avg_Salary`). Tras la derivación, eliminamos la columna redundante `Salary Estimate`.

$$\text{Avg\_Salary} = \frac{\text{Min\_Salary} + \text{Max\_Salary}}{2}$$

> 💡 **Feature Engineering:** Creación de variables sintetizadas que aumentan la interpretabilidad y reducen la complejidad en análisis posteriores.

In [ ]:
# Derivación del salario promedio
df['Avg_Salary'] = (df['Max_Salary'] + df['Min_Salary']) / 2

# Purga de la columna original (redundancia estructural)
df.drop('Salary Estimate', inplace=True, axis=1)

### 🔹 2.4 Saneamiento Textual de Entidades Corporativas (`Company Name`)

Depuramos los nombres de empresas eliminando los saltos de línea (`\n`) y el rating que venían concatenados desde la extracción web original, aislando la primera posición mediante `.str.split('\n').str[0]`.

In [ ]:
# Eliminación de artefactos de scraping (\n + rating) en Company Name
df['Company Name'] = df['Company Name'].str.split('\n', expand=True)[0]

### 🔹 2.5 Transmutación de Valores Centinela a Nulos Explícitos (`NaN`)

Estandarizamos los indicadores arbitrarios de datos ausentes (`-1`, `'-1'`, `'Unknown'`) convirtiéndolos formalmente en `np.nan`.

> ⚠️ **Principio de Integridad:** Un valor centinela numérico como `-1` en una columna como `Rating` (1 a 5) arruina la media y genera sesgos severos. Al transmutarlo a `NaN`, Pandas y NumPy lo excluyen automáticamente de los cálculos matemáticos.

In [ ]:
# Transmutación de valores centinela a NaN nativo
df.replace([-1, '-1', 'Unknown'], np.nan, inplace=True)

### 🔹 2.6 Cuantificación Real del Déficit de Información

Con los centinelas unificados bajo la representación formal `NaN`, evaluamos la severidad del déficit de datos en cada variable mediante `.isna().sum()`.

In [ ]:
# Mapa de nulidad real del DataFrame
df.isna().sum()

#### 📊 Matriz Estratégica de Tratamiento de Nulos

| Categoría de Nulidad | Variables Afectadas | Magnitud de Ausencia | Decisión / Justificación Técnica |
| :--- | :--- | :---: | :--- |
| 🔴 **Déficit Crítico** | `Competitors`, `Easy Apply` | > 75% nulos | **Purga total de columna:** Imputar induciría un sesgo artificial inaceptable. |
| 🟡 **Déficit Categórico** | `Headquarters`, `Size`, `Founded`, `Industry`, `Sector`, `Revenue`, `Type of ownership` | Moderada (5% - 20%) | **Imputación por constante:** `"No Especificado"`. Conserva varianza y no falsea categorías. |
| 🟢 **Déficit Continuo** | `Rating` | ~12% (272 registros) | **Imputación por Mediana:** Centroide robusto no sesgado por valores extremos (outliers). |
| ⚪ **Déficit Residual** | `Min_Salary`, `Max_Salary`, `Avg_Salary` | < 0.1% (1-2 registros) | **Listwise Deletion:** Eliminación directa de filas aisladas sin impacto estadístico. |

### 🔹 2.7 Reducción de Dimensionalidad — Supresión de Variables Inviables

Eliminamos `Easy Apply` y `Competitors` para evitar contaminación sintética en la base de datos relacional.

In [ ]:
# Eliminación de variables con escasez crítica (>76% nulos)
df.drop(columns=['Easy Apply', 'Competitors'], inplace=True)

# Verificación del espacio de columnas resultante
df.columns

### 🔹 2.8 Imputación Categórica Conservadora (`"No Especificado"`)

Asignamos la etiqueta `"No Especificado"` a los atributos categóricos vacíos, manteniendo la integridad referencial y permitiendo filtros SQL explícitos.

In [ ]:
# Imputación de variables categóricas con constante explícita
cols_cat = ['Headquarters', 'Size', 'Founded', 'Type of ownership',
            'Industry', 'Sector', 'Revenue']
df[cols_cat] = df[cols_cat].fillna('No Especificado')

# Verificación parcial
df.isna().sum()

### 🔹 2.9 Imputación Estadística Univariante — `Rating` (Mediana)

Calculamos la **mediana** de la distribución de calificaciones y completamos los registros faltantes.

$$\text{Mediana}(X) = Q_2 \quad \text{(Medida de tendencia central resistente a outliers)}$$

In [ ]:
# Imputación de Rating con la mediana (robusta a outliers)
mediana_rating = df['Rating'].median()
print(f'Mediana calculada: {mediana_rating}')

df['Rating'] = df['Rating'].fillna(mediana_rating)

### 🔹 2.10 Erradicación de Nulos Residuales y Reindexación

Eliminamos las filas residuales sin datos salariales (`dropna`) y reconstruimos el índice secuencial con `.reset_index(drop=True)` para garantizar contigüidad en la memoria.

In [ ]:
# Inspección del registro con salarios faltantes
df[df['Avg_Salary'].isna()]

In [ ]:
# Purga matricial definitiva + reconstrucción del índice
df = df.dropna()
df = df.reset_index(drop=True)

# Validación final: 0 nulos en todas las columnas
print(f'Dimensiones finales: {df.shape}')
print(f'\nMapa de nulidad final:')
df.isna().sum()

---

### 🏆 Balance de Transformación — Fase 2 Completada

```text
┌───────────────────────────────────────────────────────────────────────────────────────────┐
│                          AUDITORÍA DE CALIDAD DE DATOS: ANTES vs DESPUÉS                  │
└───────────────────────────────────────────────────────────────────────────────────────────┘
```

| Métrica de Calidad | Estado Crudo (Inicial) | Estado Normalizado (Final) | Impacto / Delta |
| :--- | :---: | :---: | :--- |
| 📊 **Registros Totales** | `2,253` | `2,251` | Purgados 2 registros incompletos (<0.1%) |
| 🗂️ **Columnas / Atributos** | `16` | `15` | Eliminadas 3 columnas corruptas / Creadas 3 salariales |
| 🚫 **Valores Nulos / Centinelas** | `~5,700` nulos/anomalías | **`0`** | **100% de pureza e integridad estructural** |
| 💵 **Tipo de Dato Salarial** | `object` (ej. `"$37K-$66K"`) | `float64` (ej. `37000.0`, `66000.0`) | Habilitado para modelado y agregación SQL |
| 🔢 **Índice Matricial** | Fragmentado | Secuencial Continuo (`0` a `2250`) | Optimización de acceso y serialización |

---

---
# 📕 FASE 3: Persistencia Relacional (SQLite)
> 🎯 **Objetivo:** Migrar la matriz de datos saneada desde la memoria volátil (`Pandas DataFrame`) hacia una base de datos relacional transaccional física (`SQLite3`), asegurando persistencia, reproducibilidad y compatibilidad con consultas SQL estándar.

---

### 🔹 3.1 Apertura del Canal Transaccional (`sqlite3`)

Instanciamos una conexión al archivo físico `./data/DataAnalyst.db`. Si la base de datos no existe en disco, el motor la crea en tiempo de ejecución.

In [ ]:
import sqlite3

# Establecimiento de la conexión (crea el archivo físico si no existe)
conexion = sqlite3.connect('./data/DataAnalyst.db')

### 🔹 3.2 Volcado Matricial a Tabla Relacional (`.to_sql()`)

Persistimos el DataFrame en la tabla `ofertas_laborales` mediante el método de alta velocidad `.to_sql()`:
* 🏷️ **Tabla Destino:** `ofertas_laborales`
* 🔄 **Estrategia (`if_exists='replace'`):** Idempotencia transaccional (sobrescritura controlada).
* 🚫 **Parámetro (`index=False`):** Evita la duplicación del índice interno de Pandas en el esquema relacional.

In [ ]:
# Migración del DataFrame a tabla SQL
df.to_sql('ofertas_laborales', con=conexion, if_exists='replace', index=False)

### 🔹 3.3 Auditoría de Integridad y Reconciliación de Registros

Ejecutamos una consulta SQL `SELECT COUNT(*)` mediante un cursor para certificar que el número de tuplas persistidas en disco coincida con los **2,251 registros** saneados.

In [ ]:
# Auditoría: conteo de registros en la tabla SQL
cursor = conexion.cursor()
resultado = cursor.execute('SELECT COUNT(*) FROM ofertas_laborales').fetchone()
print(f'Registros persistidos en SQLite: {resultado[0]}')

### 🔹 3.4 Cierre del Canal y Liberación de Bloqueos

Consolidamos las transacciones y cerramos la conexión para liberar los descriptores de archivo del sistema operativo.

In [ ]:
# Cierre y consolidación del archivo físico
conexion.close()

---

### 🗄️ Arquitectura de Almacenamiento Consolidada — Fase 3

```text
proyecto_auditoria_empleo/
│
├── data/
│   ├── DataAnalyst.csv      ──► Dataset original crudo (Texto plano)
│   └── DataAnalyst.db       ──► Base de Datos Relacional SQLite (Tabla: ofertas_laborales)
│
├── auditoria_mercado_laboral_tech.ipynb  ──► Pipeline reproducible
└── Planificacion y reporte de cierre/     ──► Documentación ejecutiva y auditoría
```

<div align="center">

| ✅ Estado de Persistencia | 2,251 registros consolidados en `./data/DataAnalyst.db` |
| :--- | :--- |

</div>

---

---
# 📘 FASE 4: Explotación y Análisis Exploratorio SQL (EDA)
> 🎯 **Objetivo:** Ejecutar análisis analítico-cuantitativo directamente sobre el motor SQLite mediante consultas SQL relacionales para evaluar medidas de tendencia central, dispersión salarial, distribución sectorial y polarización geográfica.

---

### 🔹 4.1 Apertura del Canal de Consulta Analítica

Reconectamos al almacén relacional `DataAnalyst.db` para inicializar el motor de consultas analíticas.

In [ ]:
import sqlite3
import pandas as pd

# Reconexión al motor SQLite
conexion = sqlite3.connect('./data/DataAnalyst.db')

# Validación: primeros 5 registros de la tabla
query_validacion = 'SELECT * FROM ofertas_laborales LIMIT 5'
df_validacion = pd.read_sql_query(query_validacion, conexion)
df_validacion

### 🔹 4.2 Medidas Globales de Tendencia Central y Dispersión

Evaluamos los parámetros estadísticos fundamentales de la variable objetivo (`Avg_Salary`):
* 📈 **Media Aritmética (`AVG`):** Nivel salarial representativo global.
* 📉 **Mínimo (`MIN`) y Máximo (`MAX`):** Límites extremos del mercado.
* 📊 **Rango de Amplitud (`MAX - MIN`):** Grado de dispersión del espectro salarial.

In [ ]:
query_salarios = """
SELECT 
    ROUND(AVG(Avg_Salary), 2)  AS Promedio_Global,
    ROUND(MIN(Avg_Salary), 2)  AS Salario_Minimo_Global,
    ROUND(MAX(Avg_Salary), 2)  AS Salario_Maximo_Global,
    ROUND(MAX(Avg_Salary) - MIN(Avg_Salary), 2) AS Rango_Salarial
FROM ofertas_laborales
"""
df_salarios = pd.read_sql_query(query_salarios, conexion)
df_salarios

#### 💡 Interpretación Ejecutiva de Parámetros Globales

* 💵 **Salario Promedio Global:** **`$72,121 USD`** — Benchmark de referencia para el perfil de Data Analyst en el mercado estadounidense.
* ↔️ **Dispersión Salarial (Rango):** **`$116,500 USD`** (de `$33,500` a `$150,000`) — Revela una marcada heterogeneidad dictada por sector, madurez empresarial y ubicación geográfica.

### 🔹 4.3 Segmentación Bivariante por Sector Industrial

Particionamos por sector con `GROUP BY Sector` e introducimos un **filtro de significancia muestral** con `HAVING COUNT(*) >= 30` para aislar resultados estadísticamente representativos.

> 🔬 **Criterio Estadístico ($n \ge 30$):** Respaldado por el *Teorema del Límite Central*, asegurando que la estimación de medias muestrales sea robusta frente al ruido de micro-sectores.

In [ ]:
query_sector = """
SELECT 
    Sector,
    ROUND(AVG(Avg_Salary), 2) AS Promedio_Sector,
    COUNT(*) AS Volumen_Ofertas
FROM ofertas_laborales
GROUP BY Sector
HAVING COUNT(*) >= 30
ORDER BY Promedio_Sector DESC
LIMIT 10
"""
df_sector = pd.read_sql_query(query_sector, conexion)
df_sector

#### 🏢 Resumen de Inteligencia Salarial por Sector

| Sector Económico | Compensación Promedio | Volumen ($n$) | Lectura Analítica del Sector |
| :--- | :---: | :---: | :--- |
| 🧬 **Biotech & Pharmaceuticals** | **`$83,106 USD`** | 33 | **Líder en compensación:** Prima por conocimiento de dominio e I+D. |
| ⚖️ **Accounting & Legal** | **`$75,221 USD`** | 43 | Alta exigencia analítica orientada a auditoría y cumplimiento normativo. |
| 💻 **Information Technology** | **`$74,247 USD`** | **570** | **Mayor empleador:** Núcleo de absorción con remuneración superior al promedio. |
| 🏥 **Health Care** | **`$72,808 USD`** | 151 | Demanda sostenida orientada a analítica clínica y operaciones. |
| 💼 **Business Services** | **`$72,273 USD`** | **523** | Segundo mayor generador de empleo en servicios y consultoría corporativa. |

### 🔹 4.4 Segmentación Espacial por Hub Geográfico (`Location`)

Evaluamos la distribución geográfica (umbral $n \ge 30$) para identificar los polos de mayor remuneración frente a los polos de absorción masiva de talento.

In [ ]:
query_ubicacion = """
SELECT 
    Location,
    ROUND(AVG(Avg_Salary), 2) AS Promedio_Ubicacion,
    COUNT(*) AS Volumen_Ofertas
FROM ofertas_laborales
GROUP BY Location
HAVING COUNT(*) >= 30
ORDER BY Promedio_Ubicacion DESC
LIMIT 10
"""
df_ubicacion = pd.read_sql_query(query_ubicacion, conexion)
df_ubicacion

#### 📍 Hallazgos de Inteligencia Geográfica

```text
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                        TOP POLOS DE ALTA COMPENSACIÓN (PREMIUM HUBS)                   │
├────────────────────────────────────────────────────────────────────────────────────────┤
│  1. 🥇 San Jose, CA     ──►  $108,855 USD  (n=38)   ──► Epicentro Silicon Valley      │
│  2. 🥈 San Francisco, CA──►  $99,273 USD   (n=119)  ──► Hub tecnológico premium       │
│  3. 🥉 San Diego, CA    ──►  $91,460 USD   (n=62)   ──► Ecosistema BioTech e innovación│
└────────────────────────────────────────────────────────────────────────────────────────┘
```

> 🏙️ **Polo de Alta Absorción Laboral:**  
> **New York, NY:** **`310 vacantes`** (Mayor volumen absoluto del dataset) con compensación promedio de **`$71,235 USD`**.

### 🔹 4.5 Concentración de Demanda por Título de Puesto (`Job Title`)

Identificamos los roles con mayor volumen de contratación para determinar la moda del mercado y detectar variantes de nomenclatura profesional.

In [ ]:
query_titulos = """
SELECT 
    "Job Title",
    COUNT(*) AS Volumen_Ofertas,
    ROUND(AVG(Avg_Salary), 2) AS Promedio_Salario
FROM ofertas_laborales
GROUP BY "Job Title"
ORDER BY Volumen_Ofertas DESC
LIMIT 10
"""
df_titulos = pd.read_sql_query(query_titulos, conexion)
df_titulos

#### 🔍 Diagnóstico de Estandarización Textual en Roles

* 📌 **Moda Dominante:** El título canónico `"Data Analyst"` acumula **404 ofertas** con una media salarial de **`$72,577 USD`**.
* ⚠️ **Fragmentación Léxica Detectada:** Se observa dispersión en la denominación de seniority:
  * Variantes de Seniority: `Senior Data Analyst` vs `Sr. Data Analyst`
  * Variantes de Sintaxis: `Junior Data Analyst` vs `Data Analyst Junior`

> 💡 **Recomendación para Futuras Iteraciones:** La aplicación de algoritmos de coincidencia difusa (*fuzzy string matching*) o un mapeo jerárquico de seniority enriquecería modelos predictivos futuros.

### 🔹 4.6 Cierre de Conexión y Liberación de Recursos

Concluimos la sesión de explotación cerrando formalmente el cursor y la conexión a la base de datos.

In [ ]:
# Cierre definitivo de la conexión SQLite
conexion.close()
print('Conexión cerrada exitosamente.')

---

## 🏛️ Acta de Cierre y Certificación del Proyecto

```text
╔═══════════════════════════════════════════════════════════════════════════════════════╗
║                                                                                       ║
║   PROYECTO: Auditoría del Mercado Laboral Tecnológico                                 ║
║   ESTADO:   ✅ FINALIZADO Y CERTIFICADO                                               ║
║                                                                                       ║
║   FASES EJECUTADAS Y AUDITADAS:                                                       ║
║     [✅] Fase 1 — Ingesta, Diagnóstico Estructural y Perfilado                        ║
║     [✅] Fase 2 — Data Wrangling, Normalización Regex e Imputación Multivariante      ║
║     [✅] Fase 3 — Persistencia Relacional y Verificación Transaccional SQLite         ║
║     [✅] Fase 4 — Análisis Exploratorio (EDA) y Extracción de Métricas SQL            ║
║                                                                                       ║
║   RESUMEN DE ENTREGABLES TÉCNICOS:                                                    ║
║     ✔ DataFrame Limpio: 2,251 instancias | 15 variables | 0 nulos                     ║
║     ✔ Almacén Físico: SQLite database (`./data/DataAnalyst.db`)                       ║
║     ✔ Banco de Consultas SQL: Segmentación sectorial, geográfica y por rol            ║
║     ✔ Documentación Técnica: Cuaderno reproducible y reportes ejecutivos              ║
║                                                                                       ║
║   FECHA DE AUDITORÍA: 17 de Agosto, 2026                                              ║
║                                                                                       ║
╚═══════════════════════════════════════════════════════════════════════════════════════╝
```

---

<div align="center">

> 🎓 *El pipeline end-to-end ha cumplido estrictamente todos los estándares de reproducibilidad, rigor estadístico y buenas prácticas de ingeniería de datos.*

</div>
